## 7. N-HiTS Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 107.8 MB/s eta 0:00:00
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import NHiTSPredictor, HORIZONS

MODEL_NAME = 'NHiTS'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'nhits'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nhits_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nhits_val_stability_summary.json'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

val_participant_ids = load_participant_ids('val')
test_participant_ids = load_participant_ids('test')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [4]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Validation Results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = NHiTSPredictor(context_length=sample_cgm.shape[1])

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'nhits' / f'nhits_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'nhits_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Validation Results — NHiTS | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NHiTSPredictor  params: 70,002


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.85 | MAE 7.29 | gRMSE 11.74 | Zone A 96.71% | A+B 99.71%
 30 min | RMSE 17.43 | MAE 11.61 | gRMSE 19.58 | Zone A 89.90% | A+B 99.35%
 60 min | RMSE 24.95 | MAE 16.53 | gRMSE 29.48 | Zone A 81.63% | A+B 98.66%
 90 min | RMSE 29.30 | MAE 19.61 | gRMSE 35.31 | Zone A 76.59% | A+B 98.18%
120 min | RMSE 31.89 | MAE 21.74 | gRMSE 38.76 | Zone A 73.04% | A+B 97.92%

--------------------------------------------------------------------------------
Validation Results — NHiTS | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NHiTSPredictor  params: 70,002


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.95 | MAE 7.50 | gRMSE 11.84 | Zone A 96.54% | A+B 99.68%
 30 min | RMSE 17.50 | MAE 11.82 | gRMSE 19.68 | Zone A 89.76% | A+B 99.34%
 60 min | RMSE 24.99 | MAE 16.72 | gRMSE 29.50 | Zone A 81.68% | A+B 98.69%
 90 min | RMSE 29.31 | MAE 19.83 | gRMSE 35.30 | Zone A 76.53% | A+B 98.24%
120 min | RMSE 31.88 | MAE 22.05 | gRMSE 38.75 | Zone A 72.67% | A+B 97.89%

--------------------------------------------------------------------------------
Validation Results — NHiTS | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NHiTSPredictor  params: 70,002


...

 15 min | RMSE 10.81 | MAE 7.21 | gRMSE 11.64 | Zone A 96.76% | A+B 99.74%
 30 min | RMSE 17.41 | MAE 11.51 | gRMSE 19.49 | Zone A 90.04% | A+B 99.34%
 60 min | RMSE 24.94 | MAE 16.61 | gRMSE 29.41 | Zone A 81.62% | A+B 98.67%
 90 min | RMSE 29.26 | MAE 19.68 | gRMSE 35.19 | Zone A 76.56% | A+B 98.19%
120 min | RMSE 31.85 | MAE 21.86 | gRMSE 38.65 | Zone A 72.91% | A+B 97.94%

--------------------------------------------------------------------------------
Validation Results — NHiTS | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NHiTSPredictor  params: 70,002


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.90 | MAE 7.30 | gRMSE 11.87 | Zone A 96.77% | A+B 99.74%
 30 min | RMSE 17.49 | MAE 11.55 | gRMSE 19.82 | Zone A 90.02% | A+B 99.32%
 60 min | RMSE 24.99 | MAE 16.47 | gRMSE 29.77 | Zone A 82.02% | A+B 98.65%
 90 min | RMSE 29.28 | MAE 19.55 | gRMSE 35.49 | Zone A 77.03% | A+B 98.20%
120 min | RMSE 31.85 | MAE 21.67 | gRMSE 38.90 | Zone A 73.31% | A+B 97.95%

--------------------------------------------------------------------------------
Validation Results — NHiTS | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: NHiTSPredictor  params: 70,002


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 10.80 | MAE 7.23 | gRMSE 11.69 | Zone A 96.84% | A+B 99.72%
 30 min | RMSE 17.40 | MAE 11.55 | gRMSE 19.55 | Zone A 90.02% | A+B 99.34%
 60 min | RMSE 24.99 | MAE 16.64 | gRMSE 29.46 | Zone A 81.76% | A+B 98.66%
 90 min | RMSE 29.28 | MAE 19.76 | gRMSE 35.26 | Zone A 76.62% | A+B 98.18%
120 min | RMSE 31.83 | MAE 21.91 | gRMSE 38.70 | Zone A 72.75% | A+B 97.91%

--------------------------------------------------------------------------------


In [5]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [6]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [7]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== NHiTS validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 10.88 +/- 0.06 (2.26) mg/dL   MAE = 7.31 +/- 0.09 (1.48)   gRMSE = 11.81 +/- 0.11 (3.10)   Zone A = 96.75 +/- 0.11 (3.17)%   Zone A+B = 99.71 +/- 0.02 (0.59)%
 30-min:  RMSE = 17.46 +/- 0.04 (3.74) mg/dL   MAE = 11.60 +/- 0.09 (2.37)   gRMSE = 19.69 +/- 0.14 (5.65)   Zone A = 89.97 +/- 0.12 (5.34)%   Zone A+B = 99.34 +/- 0.02 (0.95)%
 60-min:  RMSE = 24.96 +/- 0.03 (7.47) mg/dL   MAE = 16.56 +/- 0.13 (4.79)   gRMSE = 29.55 +/- 0.12 (11.24)   Zone A = 81.80 +/- 0.25 (5.23)%   Zone A+B = 98.67 +/- 0.01 (1.46)%
 90-min:  RMSE = 29.27 +/- 0.02 (10.38) mg/dL   MAE = 19.66 +/- 0.15 (6.90)   gRMSE = 35.33 +/- 0.10 (15.33)   Zone A = 76.72 +/- 0.32 (7.58)%   Zone A+B = 98.19 +/- 0.02 (1.98)%
120-min:  RMSE = 31.84 +/- 0.04 (12.15) mg/dL   MAE = 21.80 +/- 0.18 (8.33)   gRMSE = 38.75 +/- 0.10 (18.01)   Zone A = 72.97 +/- 0.45 (8.32)%   Zone A+B = 97.92 +/- 0.03 (2.40)%


In [8]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nhits/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nhits/nhits_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nhits/nhits_stability_summary.json
